# multi_15m_strat_041

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (40).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `15m` |
| Code cells | 9 |
| Detected functions | resample_1hr_custom, compute_kama_numba, compute_rs_vol_numba, compute_kama_indicator, process_ticker, backtest_trades, calculate_metrics, apply_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'AXISBANK',
       'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BEL', 'BHARTIARTL',
       'BPCL', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT',
       'GRASIM', 'HCLTECH', 'HEROMOTOCO', 'HINDALCO', 'HINDUNILVR',
       'ICICIBANK', 'INDUSINDBK', 'JSWSTEEL', 'KOTAKBANK',
       'MARUTI', 'MM', 'NTPC', 'ONGC', 'SBILIFE', 'SBIN', 'SHRIRAMFIN',
       'TATAMOTORS', 'TATASTEEL', 'TECHM', 'TITAN',
       'TRENT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h', offset = '15min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
from numba import njit

# ============================================
# Numba Accelerated Functions
# ============================================

@njit
def compute_kama_numba(price, sc):
    kama = np.empty(len(price))
    kama[0] = price[0]
    for i in range(1, len(price)):
        if np.isnan(sc[i]):
            kama[i] = kama[i - 1]
        else:
            kama[i] = kama[i - 1] + sc[i] * (price[i] - kama[i - 1])
    return kama

@njit
def compute_rs_vol_numba(open_, high, low, close):
    out = np.empty(len(open_))
    out[:] = np.nan
    for i in range(len(open_)):
        if open_[i] == 0 or close[i] == 0:
            continue
        log_hc = np.log(high[i] / close[i])
        log_ho = np.log(high[i] / open_[i])
        log_lc = np.log(low[i] / close[i])
        log_lo = np.log(low[i] / open_[i])
        out[i] = log_hc * log_ho + log_lc * log_lo
    return out

# ============================================
# KAMA Signal Generation Function
# ============================================

def compute_kama_indicator(
    df,
    kama_length=3,
    fast_ema=2,
    slow_ema=50,
    norm_period=70,
    upper_thresh=2,
    lower_thresh=-2,
    rs_period=14,
    rs_percentile=50
):
    price = df['close'].values
    change = np.abs(np.concatenate([[np.nan] * kama_length, price[kama_length:] - price[:-kama_length]]))
    volatility = pd.Series(np.abs(np.diff(price, prepend=np.nan))).rolling(kama_length).sum().values
    efficiency_ratio = change / np.where(volatility == 0, np.nan, volatility)

    fast_sc = 2 / (fast_ema + 1)
    slow_sc = 2 / (slow_ema + 1)
    sc = (efficiency_ratio * (fast_sc - slow_sc) + slow_sc) ** 2

    kama_values = compute_kama_numba(price, sc)
    kama_series = pd.Series(kama_values, index=df.index)

    mean = kama_series.rolling(norm_period).mean()
    std = kama_series.rolling(norm_period).std()
    normalized_kama = (kama_series - mean) / std

    open_, high, low, close = df['open'].values, df['high'].values, df['low'].values, df['close'].values
    rs_vol_raw = compute_rs_vol_numba(open_, high, low, close)
    rs_vol = pd.Series(rs_vol_raw).rolling(rs_period).mean()
    rs_thresh = rs_vol.rolling(norm_period).quantile(rs_percentile / 100.0)

    long_condition = (normalized_kama > upper_thresh) & (rs_vol > rs_thresh)
    short_condition = (normalized_kama < lower_thresh) & (rs_vol > rs_thresh)

    signal = pd.Series(0, index=df.index)
    signal[long_condition] = 1
    signal[short_condition] = 2

    df['kama'] = kama_series
    df['normalized_kama'] = normalized_kama
    df['rs_vol'] = rs_vol
    df['rs_thresh'] = rs_thresh
    df['Signal'] = signal.shift(1)

    return df

# ============================================
# Trade Simulation Function
# ============================================

def process_ticker(
    ticker, ticker_data, initial_capital, brokerage, slippage,
    tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time
):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp = row['Date']
        close = row['close']
        open_price = row['open']
        high = row['high']
        low = row['low']
        signal = row['Signal']
        time_str = pd.to_datetime(timestamp).strftime('%H:%M')

        # Check for open position
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif time_str >= end_time:
                exit_price, exit_reason = close, "EOD Close"
            else:
                continue

            exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
            raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
            pnl = raw_pnl - (brokerage * initial_capital * 2)
            cumulative_pnl += pnl

            trades.append([
                ticker, position["entry_time"], timestamp, entry_price, exit_price,
                pnl, exit_reason, cumulative_pnl, direction
            ])
            del open_positions[ticker]
            continue

        # Open new position
        if start_trading_time < time_str < stop_trading_time and ticker not in open_positions:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue
            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {
                "entry_time": timestamp,
                "entry_price": entry_price,
                "direction": direction,
                "qty": qty
            }

    return trades

# ============================================
# Backtest Executor
# ============================================

def backtest_trades(data, tp_pct, sl_pct):
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    end_time = "15:10"
    stop_trading_time = "14:20"
    start_trading_time = "09:10"
    unique_tickers = data['Ticker'].unique()

    data_dict = {ticker: df.to_dict('records') for ticker, df in data.groupby('Ticker')}

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker, data_dict[ticker], initial_capital, brokerage, slippage,
            tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time
        ) for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# ============================================
# Run Backtest
# ============================================
# Step 1: Generate signals
data = compute_kama_indicator(data)


tradebook = backtest_trades(data, tp_pct=0.03, sl_pct=0.001)
tradebook.to_csv("/content/drive/MyDrive/tradebook_filtered.csv", index=False)

# Preview
tradebook


In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# === Metrics Calculation Function (with R² and Monthly Consistency) ===
def calculate_metrics(tradebook):
    if tradebook.empty:
        return 0, 0, 0, 0

    # Ensure datetime
    tradebook['Date'] = pd.to_datetime(tradebook['Entry Time'])  # Replace with correct column if different

    equity = tradebook['PnL'].cumsum()
    total_profit = equity.iloc[-1]

    roll_max = equity.cummax()
    drawdown = roll_max - equity
    max_drawdown = drawdown.max()

    if max_drawdown == 0:
        ratio = float('inf') if total_profit > 0 else 0
    else:
        ratio = total_profit / max_drawdown

    # R² for equity curve smoothness
    x = np.arange(len(equity)).reshape(-1, 1)
    y = equity.values.reshape(-1, 1)
    model = LinearRegression().fit(x, y)
    r2 = model.score(x, y)

    # Monthly return consistency metric
    tradebook['Month'] = tradebook['Date'].dt.to_period('M')
    monthly_returns = tradebook.groupby('Month')['PnL'].sum()
    if len(monthly_returns) < 2:
        monthly_consistency = 0
    else:
        monthly_mean = monthly_returns.mean()
        monthly_std = monthly_returns.std()
        monthly_consistency = monthly_mean / monthly_std if monthly_std > 0 else float('inf')

    return ratio, total_profit, r2, monthly_consistency

# === Hyperparameter Grid ===
param_grid = {
    'kama_length': [5],
    'fast_ema': [2, 3],
    'slow_ema': [30, 50, 80],
    'norm_period': [70, 80, 90],
    'upper_thresh': [1.5, 2, 2.5],
    'lower_thresh': [-1.5, -2, -2.5],
    'rs_period': [10, 14],
    'rs_percentile': [10, 30, 50],
    'tp_pct': [0.03, 0.05],
    'sl_pct': [0.001, 0.01]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    print(f"\n=== [{i}/{len(param_combinations)}] Testing Parameters ===")
    for k, v in param.items():
        print(f"{k}: {v}")

    temp_data = data.copy()

    def apply_strategy(group):
        return compute_kama_indicator(
            group,
            kama_length=param['kama_length'],
                fast_ema=param['fast_ema'],
                slow_ema=param['slow_ema'],
                norm_period=param['norm_period'],
                upper_thresh=param['upper_thresh'],
                lower_thresh=param['lower_thresh'],
                rs_period=param['rs_period'],
                rs_percentile=param['rs_percentile']
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        sl_pct=param['sl_pct'],
        tp_pct=param['tp_pct']
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency = calculate_metrics(tradebook)

    print(f"➡️ Profit-to-Drawdown Ratio: {ratio:.4f}")
    print(f"➡️ Total Profit: ₹{total_profit:,.2f}")
    print(f"➡️ Equity Curve R²: {r2:.4f}")
    print(f"➡️ Monthly Consistency Score: {monthly_consistency:.4f}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.6 +
    results_df['Equity R2'] * 0.2 +
    results_df['Monthly Consistency'] * 0.2
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())
